# gotooltrain — 4B tam fine-tune (Colab)

`Qwen/Qwen3.5-4B` → Go + tool-use modeli, tam fine-tune, **periyodik Hugging Face Hub
yüklemesiyle**.

Bu defter şunu yapar, başka hiçbir şeyi:

1. Depoyu remote'dan çeker (`alexzakarov/model-trainer`).
2. `pip install -e ".[dev,train]"` ile kurar.
3. **Go ve rtk toolchain'lerini** sabitlenmiş digest'lerle kurar (Colab'da ikisi de yok).
4. Go-UT-Bench korpusunu indirip ölçer (lisans süzgeci dahil).
5. Token formatını **gerçek Qwen3.5-4B tokenizer'ıyla** doğrular — maske yanlışsa
   burada durur, GPU saatleri harcanmaz.
6. 4B tam fine-tune'u başlatır ve **her N adımta** checkpoint'ı Hub'a gönderir.

---

## ⚠️ `REPO_REF`'i sabitle

`REPO_REF` şu an `main`. Bu **bir kez çalışmak** içindir: yarın aynı defteri
çalıştırmak farklı kod demektir. `95b1aa3` gibi bir commit sha'sına sabitle,
böylece aynı komut aynı kodu çeker.

## ⚠️ Token'ın yazma yetkisi

Hedef depo 1. hücredeki `HF_REPO_ID`. `HF_TOKEN`'ın o depoya **yazma** yetkisi
olmalı. Depo şu an boş; ilk push onu doldurur. `upload_folder` model card yazmaz
— kart elle eklenir.

## ⚠️ GPU seçimi

4B **tam** fine-tune AdamW ile ~18 GB (ağırlık + gradyan) ister. Colab'da:

| Runtime | VRAM | Bu defter |
|---|---|---|
| T4 (ücretsiz) | 15 GB | ❌ çalışmaz |
| L4 | 24 GB | ⚠️ Adafactor + kısa bağlamla sığar |
| **A100 40 GB** | 40 GB | ✅ önerilen |
| A100 80 GB | 80 GB | ✅ |

Aşağıdaki hücre yetersiz VRAM'i sessizce geçmez — adını ve nedenini söyler.

In [ ]:
# @title 1 — Parameters
#
# Buradaki değerler mock olarak girdi. Gerçek koşuda HF_TOKEN'ı Colab Secrets'a
# (🔑 soldaki panel) koy; buraya yazmak token'ı notebook çıktısına ve git geçmişine
# sızdırır.

REPO_URL = 'https://github.com/alexzakarov/model-trainer.git'
REPO_REF = "main"                      # commit sha da olabilir; sha daha tekrarlanabilir

MODEL_ID = "Qwen/Qwen3.5-4B"          # eğitilecek taban model
HF_REPO_ID = 'alexzakkarov/qwen3.5-golang'    # hedef Hub deposu
HF_TOKEN = "hf_mock_replace_me"        # mock — Colab Secrets'tan okunacak
DRY_RUN = False                        # True: takvimi prova et, hiçbir şey yükleme

# Eğitim
#
# 4096, varsayılan 32768 değil — ölçülmüş bir seçim. Qwen3.5-4B'de 32 katmanın
# 24'ü Gated DeltaNet (lineer dikkat) ve durum [32 v_heads, 128, 128] yani
# **token başına 1 MB**. Bu yüzden aktivasyon belleği bağlamla **doğrusal** büyür:
# 8192 token'da checkpointing olmadan ~196 GB. 80 GB kartta 8192 denendi ve
# OOM verdi. Aşağıdaki dağılım bu korpusun gerçek uzunlukları (ölçüldü):
#
#     < 2048 : %11     < 4096 : %42     < 8192 : %72     medyan 4848
#
# 4096 hem sığdırıyor hem korpusun yarısını bırakıyor. 8192'yi denemek istersen
# önce DRY_RUN=True ile bak: tek adım sığmıyorsa saatler harcanmış olur.
CONTEXT_LENGTH = 4096
EXPANDABLE_SEGMENTS = True               # parçalanmayı azaltır (bkz. aşağıdaki hücre)
EPOCHS = 1
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 1e-5
MAX_RECORDS = 400                      # korpus 4491 kayıt; süreyi sınırlamak için kes
MAX_TOKENS_PER_RECORD = CONTEXT_LENGTH  # üstünü atılır, *sayısı raporlanır*

# Yayınlama
PUSH_EVERY = 25                        # optimizer adımı

print(f"repo   : {REPO_URL}@{REPO_REF}")
print(f"model  : {MODEL_ID}")
print(f"push to: {HF_REPO_ID} every {PUSH_EVERY} steps (dry_run={DRY_RUN})")

In [ ]:
# @title 2 — GPU ön kontrolü
#
# Yetersiz VRAM'de çalıştırmak, OOM ile 40 dakika sonra ölmekten iyidir ama yine de
# kötüdür: ne öğrenildiği belirsiz. Adı ve gerekeni söyleyip duruyoruz.

import subprocess

import torch

GB = 1024**3
free = torch.cuda.is_available()
total_gb = (torch.cuda.get_device_properties(0).total_memory / GB) if free else 0.0
MIN_GB = 22.0  # Adafactor + gradient checkpointing + 8K bağlam için gerçek taban

print(f"torch      : {torch.__version__}")
print(f"cuda       : {free}")
print(f"device     : {torch.cuda.get_device_name(0) if free else 'none'}")
print(f"vram       : {total_gb:.1f} GB (gereken >= {MIN_GB:.0f} GB)")

if total_gb < MIN_GB:
    raise SystemExit(
        f"{total_gb:.1f} GB VRAM bu tam fine-tune için yetersiz. "
        "Runtime > Change runtime type > A100 (40 GB) seçin. "
        "T4'de 4B tam fine-tune matematiksel olarak sığmaz; LoRA'ya düşmenin "
        "bu projenin kararı değil, ayrı bir karar olurdu."
    )

In [ ]:
# @title 3 — Depoyu çek ve kur
#
# REF bir commit sha ise tam tekrarlanabilir olur. 'main' daha basit ama yarın
# aynı defteri çalıştırmak farklı kod demektir.

import os
import subprocess
import sys

os.environ["GIT_TERMINAL_PROMPT"] = "0"  # şifre soran bir clone sessizce beklemesin

def run(*args: str) -> str:
    """Bir komutu çalıştır; stdout döndür, hata olursa mesajla dur."""
    result = subprocess.run(args, capture_output=True, text=True, check=False)
    if result.returncode != 0:
        raise SystemExit(f"{' '.join(args)} failed:\n{result.stdout}\n{result.stderr}")
    return result.stdout.strip()

target = "/content/model-trainer"
if os.path.isdir(target):
    run("git", "-C", target, "fetch", "--all")
    run("git", "-C", target, "checkout", REPO_REF)
    run("git", "-C", target, "pull", "--ff-only", "origin", REPO_REF)
else:
    run("git", "clone", REPO_URL, target)
    run("git", "-C", target, "checkout", REPO_REF)

print(run("git", "-C", target, "log", "-1", "--format=%H %s"))

# transformers 5.x şart: Qwen3.5 config'i model_type qwen3_5 bildiriyor ve 4.x onu
# reddediyor (tokenizer yüklenir, config yüklenmez). Sürüm burada da sabitleniyor,
# çünkü Colab'ın imajı zamanla değişir.
run(sys.executable, "-m", "pip", "install", "-q", "-U", "pip")
run(sys.executable, "-m", "pip", "install", "-q", "transformers>=5.17,<6")
run(sys.executable, "-m", "pip", "install", "-q", "-e", f"{target}[dev,train]")

os.chdir(target)
print("cwd:", os.getcwd())

# ---------------------------------------------------------------- kernel path
#
# `pip install -e` yalnızca *yeni* süreçlerde görünür. .pth ve editable-finder
# çengelleri yorumlayıcı **başlangıcında** kurulur; Colab çekirdeği ise bu
# hücreden çok önce başlamıştır. Yani alt süreçler (kalite kapısı, eğitim koşusu)
# paketi görür, çekirdek görmez — ve hata beş hücre sonra, başka bir hücrenin
# satırında belirir. Burada ölçtüm: pip'i çalıştıran yorumlayıcı kendi kurduğu
# paketi import edemiyor.
#
# Bu yüzden kaynak dizini çekirdeğin sys.path'ine de ekleniyor. Gizli bir
# düşüş değil: kurulum zaten yapıldı, bu yalnızca aynı kurulumu *bu* sürece
# görünür kılıyor. Konsol betikleri (gotooltrain-train vb.) alt süreçlerde çalışır
# ve zaten yolunu bulur.
sys.path.insert(0, os.path.join(target, "src"))

import gotooltrain  # noqa: E402

print("gotooltrain:", gotooltrain.__version__, "->", gotooltrain.__file__)

# Klonun içinden geldiğini doğrula. Path'te başka bir gotooltrain varsa (sık
# gelen bir tuzak: daha önce kurulmuş bir sürüm) sessizce onu eğitirdik.
if not gotooltrain.__file__.startswith(target):
    raise SystemExit(
        f"gotooltrain {gotooltrain.__file__} konumundan geliyor, klon {target} değil. "
        "Sistemde başka bir kurulum var ve eğitimi yanlış kodla yapacağız."
    )

In [ ]:
# @title 4 — Go ve rtk toolchain'leri
#
# Colab'da ikisi de yok. Bu bir eksik değil, **kapının kırılma biçimi**: katalogdaki
# komutların çoğu rtk üzerinden çalışır (`rtk go test`, `rtk read`, `rtk grep`), ve
# paketin kendi testleri gerçek komutları koşar. rtk yokken `go_test` "rtk not found"
# ile HARNESS_ERROR döner ve hata **kırık paket** gibi görünür — oysa kırık olan
# ortamdır.
#
# İkisi de sabit + doğrulanmış. rtk'nin sürümü ve digest'i docker/go-sandbox/Dockerfile'ın
# sabitlediğiyle **aynı**: yani defter ile sandbox imajı kanıtlanabilir biçimde aynı
# binary'yi koşuyor. Katalogun çıktı formatı rtk'nin formatıdır; farklı bir build
# üzerinde ölçmek, eval harness'ın ölçeceği formatı ölçmemek demektir.
#
# rtk doğrulaması çift kaynaklı: gömülü sabit *ve* yayıncının checksums.txt'i. Uyuşmazsa
# kurulum iptal edilir.
#
# GOTOOLCHAIN=local: go.mod'daki `go 1.23` direktifinin ağdan toolchain indirmesini
# engeller — testlerin ürettiği modüller ağ olmadan da derlenir.

import hashlib
import io
import os
import pathlib
import platform
import shutil
import tarfile
import urllib.request

GO_VERSION = '1.23.6'
GO_SHA256 = '9379441ea310de000f33a4dc767bd966e72ab2826270e038e78b2c53c2e7802d'
GO_ROOT = '/usr/local/go'

RTK_VERSION = '0.50.0'
RTK_SHA256 = 'bc2b8902b0d9c796c82ef45f16ae2307e17757afeca5ee156235a3dc7bda5f89'
RTK_BASE_URL = 'https://github.com/rtk-ai/rtk/releases/download'
RTK_ASSET = 'rtk-x86_64-unknown-linux-musl.tar.gz'


def fetch_verified(url: str, expected_sha256: str, what: str) -> bytes:
    """İndir, hash'le, sabitle karşılaştır. Uyuşmazsa kurulum yapılmaz."""
    print(f"  indiriliyor {what}: {url}")
    with urllib.request.urlopen(url, timeout=300) as response:
        payload = response.read()
    digest = hashlib.sha256(payload).hexdigest()
    print(f"  sha256           {digest}")
    if digest != expected_sha256:
        raise SystemExit(
            f"{what} doğrulanamadı. İndirilen {digest}, beklenen {expected_sha256}. "
            "Kurulum iptal edildi — doğrulanmamış bir toolchain, hiç kurmamaktan kötüdür."
        )
    return payload


if platform.machine() not in ("x86_64", "AMD64"):
    raise SystemExit(
        f"bu defter {platform.machine()} mimarisi için; rtk'nin sabitlenmiş build'i "
        "x86_64. Colab'da T4/L4/A100 seçtiğiniz için bu burada sorun olmaz."
    )

# ------------------------------------------------------------------ Go
print("Go kuruluyor...")
go_archive = f"go{GO_VERSION}.linux-amd64.tar.gz"
go_payload = fetch_verified(f"https://go.dev/dl/{go_archive}", GO_SHA256, "go")

root = pathlib.Path(GO_ROOT)
if root.exists():
    shutil.rmtree(root)
with tarfile.open(fileobj=io.BytesIO(go_payload)) as tar:
    try:
        tar.extractall(root.parent, filter="data")
    except TypeError:  # Python < 3.12 has no filter argument
        tar.extractall(root.parent)

# ------------------------------------------------------------------ rtk
print("rtk kuruluyor...")
base = f"{RTK_BASE_URL}/v{RTK_VERSION}"
rtk_payload = fetch_verified(f"{base}/{RTK_ASSET}", RTK_SHA256, "rtk")

# İkinci kaynak: yayıncının kendi checksum listesi. Sabit "neyi inceledik"i söyler,
# bu indirme "yayıncı hâlâ öyle diyor"u kanıtlar.
with urllib.request.urlopen(f"{base}/checksums.txt", timeout=120) as response:
    published = response.read().decode("utf-8", "replace")
expected_line = next(
    (line for line in published.splitlines() if line.split()[-1:] == [RTK_ASSET]), None
)
if expected_line is None:
    raise SystemExit(f"checksums.txt içinde {RTK_ASSET} yok; yayıncı listeyi değiştirmiş.")
if expected_line.split()[0] != RTK_SHA256:
    raise SystemExit(
        f"sabit ile yayıncı uyuşmuyor. Gömülü {RTK_SHA256}, yayıncı "
        f"{expected_line.split()[0]}. İkisi de doğru olmalı."
    )
print(f"  yayıncı da doğruladı: {expected_line.split()[0][:16]}...")

rtk_dir = pathlib.Path("/tmp/rtk-dist")
if rtk_dir.exists():
    shutil.rmtree(rtk_dir)
rtk_dir.mkdir(parents=True)
with tarfile.open(fileobj=io.BytesIO(rtk_payload)) as tar:
    tar.extractall(rtk_dir)
binary = rtk_dir / "rtk"
shutil.copy2(binary, "/usr/local/bin/rtk")
os.chmod("/usr/local/bin/rtk", 0o755)
shutil.rmtree(rtk_dir, ignore_errors=True)

# ------------------------------------------------------------------ PATH
# os.environ üzerinden, shell export'u değil: subprocess os.environ'ı miras alır,
# dolayısıyla hem kalite kapısı hem eğitim koşusu aynı toolchain'i görür. Shell'e
# kurulup sürece aktarılmayan bir Go, kapıdan geçer ve eğitimde patlar.
os.environ["PATH"] = f"{root / 'bin'}:/usr/local/bin:{os.environ['PATH']}"
os.environ["GOTOOLCHAIN"] = "local"

# Parçalanma ayarı. CUDA önbelleğinde ayrılmış ama kullanılmayan bloklar uzun
# bir koşuda birikir ve "boş" görünen belleği yer. Bu ayar ayrılmış blokları
# büyüterek birleştirir; etkisi çalışma zamanında görülür, tahmin edilemez.
if EXPANDABLE_SEGMENTS:
    _existing = os.environ.get("PYTORCH_CUDA_ALLOC_CONF", "")
    if "expandable_segments" not in _existing:
        os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
            f"{_existing},expandable_segments:True" if _existing else "expandable_segments:True"
        )
    print("PYTORCH_CUDA_ALLOC_CONF:", os.environ["PYTORCH_CUDA_ALLOC_CONF"])

# `run` 3. hücreden.
print(run("go", "version"))
print(run("rtk", "--version"))

In [ ]:
# @title 5 — Token: Colab secret ya da mock
#
# Proje kuralı: sessiz düşüş yok. Token yoksa yayınlama başlamadan hata verir —
# çünkü 700 adım sonra öğrenmek, 700 adım önce öğrenmekten pahalıdır.

import os

try:
    from google.colab import userdata

    HF_TOKEN = userdata.get("HF_TOKEN")
    print("HF_TOKEN: Colab secret'tan okundu")
except (ImportError, Exception):  # noqa: B014 - colab dışında da çalışabilmeli
    print("Colab secret yok; hücre 1'deki mock değer kullanılacak")

if not HF_TOKEN or HF_TOKEN == "hf_mock_replace_me":
    if DRY_RUN:
        os.environ["HF_TOKEN"] = HF_TOKEN or "hf_mock_dry_run"
        print("DRY_RUN=True: gerçek bir token gerekmiyor")
    else:
        raise SystemExit(
            "HF_TOKEN yok. Sol panelden 🔑 Secrets'a HF_TOKEN ekleyin, ya da "
            "DRY_RUN=True ile takvimi hiçbir şey yüklemeden prova edin."
        )
else:
    os.environ["HF_TOKEN"] = HF_TOKEN

from huggingface_hub import login

login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("hub client hazır")

In [ ]:
# @title 6 — Kalite kapısı (paket kendi testini koşar)
#
# Colab'a kurduğumuz şeyin bu depodakiyle aynı şey olduğunu doğrulamadan 4B
# eğitimi başlatmak, 4B eğitimi başlatmadan önce yapılabilecek en pahalı kontrolü
# atlamak olurdu. Testler ~2 dakika sürer; 4B bir epoch saatler sürer.
#
# -x: ilk kırıklıkta dur. Kapının amacı "bir şeyler yanlış" demek, listelemek değil.
# Docker ve rtk testleri yoksa *atlanır* (gerekçeleri skip metninde) — bu, eksik
# bağımlılığın sessizce yeşile dönmesi değil, açıkça raporlanmasıdır. Son satırda
# kaç testin atlandığını ve nedenini görüyorsunuz.

import subprocess

started = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "-x", "-rs"],
    cwd=target,
    capture_output=True,
    text=True,
    check=False,
)
print(started.stdout[-3000:])
print(started.stderr[-2000:])

skipped = [line for line in started.stdout.splitlines() if line.startswith("SKIPPED")]
if skipped:
    print(f"\n--- atlanan testler ({len(skipped)}) ve gerekçeleri ---")
    for line in skipped:
        print(" ", line)

if started.returncode != 0:
    raise SystemExit(
        "Kalite kapısı kırmızı. Eğitime başlamak, kırık bir ağaca kanat takmak olurdu. "
        "Yukarıdaki çıktıya bakın. "
        f"(Atlanan test: {len(skipped)} — bir bağımlılık eksikse bu normal; "
        "kırmızı bir test normal değil.)"
    )
print("kapı yeşil")

In [ ]:
# @title 7 — Go korpusunu indir, süz, ölç
#
# `go-pairs` lisans süzgecini uygular: Go-UT-Bench "permissive" diyor ama
# terraform BUSL-1.1, go-ethereum LGPL-3.0. İkisi de varsayılan olarak reddedilir
# ve rapora yazılır. Ayrıca split'ler örtüştüğü için tekilleştirilir.

import subprocess

run(
    sys.executable, "-m", "gotooltrain.datacli", "go-pairs",
    "--download-to", "data/go-ut-bench",
    "--out", "data/go-dapt.jsonl",
    "--unit-tests-out", "data/go-unit-tests.jsonl",
    "--report", "data/go-dapt-report.json",
)
print(open("data/go-dapt-report.json", encoding="utf-8").read()[:1200])

In [ ]:
# @title 8 — SFT verisini ölç, süz, doğrula
#
# Buradaki veri Go-UT-Bench'in *birim testi yaz* görevleridir; değerlendirme
# trajektöri değil. Bu, yetenek ölçümü değil — format ve boru hattı denemesidir.
# Tek turlu, araçsız, sekiz depodan biri ağırlıklı bir korpus katalogu öğretmez;
# `measure` bunu "yetersiz" diye bildirecek ve bu doğru cevaptır. Değerlendirme
# için konteyner havuzu, rtk ve gerçek Go depoları gerekir (docs/EVAL.md).
#
# **Burada token id yazmıyoruz.** Eğitim CLI'si mesajları kendisi render eder;
# tek render yolu ilkesi budur. Buradaki iş render etmek değil **ölçmek**:
# hangi kayıtların bağlama sığdığını ve maskenin doğru olduğunu, pahalı
# hücrelerden önce görmek.
#
# Uzunluk **katalogla** ölçülüyor, çünkü eğitim CLI'sı da katalogla render edecek.
# Katalogsuz ölçseydik altımızda kalırdı ve bağlam dışı bir kayıt eğitimi düşürürdü.

import json

from transformers import AutoTokenizer

from gotooltrain import (
    catalog,
    install_template,
    load_template_source,
    normalize_conversation,
    read_jsonl,
    render_example,
)

TOOLS = catalog()
tokenizer = install_template(AutoTokenizer.from_pretrained(MODEL_ID), load_template_source())
print("tokenizer hazır:", type(tokenizer).__name__)

records = list(read_jsonl("data/go-unit-tests.jsonl"))[:MAX_RECORDS]
print(f"{len(records)} kayıt okundu (sınır: {MAX_RECORDS})")

kept = []
first_rendered = None
dropped_invalid = 0
dropped_truncated = 0
dropped_no_supervision = 0
total_tokens = 0
supervised_tokens = 0
longest = 0

for record in records:
    try:
        conversation = normalize_conversation(record["messages"], TOOLS)
    except Exception as exc:  # noqa: BLE001 - her ret sayılır, ilk birkaçı adlandırılır
        dropped_invalid += 1
        if dropped_invalid <= 3:
            print(f"  reddedildi (geçersiz): {type(exc).__name__}: {exc}")
        continue
    example = render_example(tokenizer, conversation, max_length=MAX_TOKENS_PER_RECORD)

    # **Tavana oturan kayıt kesilmiştir.** Kesme baştaki token'ları korur, yani
    # asistan turn'ü sınırdan önce başladıysa *denetimli token hâlâ vardır* ve
    # "denetim var mı" sorusu kesilmiş bir kaydı geçirir. Oysa eğitim kaydı
    # sınırsız yeniden render eder ve `assert_examples_fit` onu bağlam dışı
    # diye reddeder — yani ön kontrol, eğitimin reddedeceği kaydı onaylıyordu.
    # (Ölçüldü: ilk 400 kaydın 74'ü bu durumdaydı; gerçek uzunluklar 8.5K–17.6K.)
    if len(example.input_ids) >= MAX_TOKENS_PER_RECORD:
        dropped_truncated += 1
        continue
    if example.supervised_tokens == 0:
        # Asistan turn'üne hiç ulaşılamadı: denetlenecek bir şey yok.
        dropped_no_supervision += 1
        continue
    kept.append({"messages": record["messages"], "tools": TOOLS})
    total_tokens += len(example.input_ids)
    supervised_tokens += example.supervised_tokens
    longest = max(longest, len(example.input_ids))
    if first_rendered is None:
        first_rendered = example

if not kept:
    raise SystemExit(
        f"{len(records)} kaydın hiçbiri kullanılabilir değil "
        f"({dropped_truncated} kesilmiş, {dropped_no_supervision} denetimsiz, "
        f"{dropped_invalid} geçersiz). Eğitilecek veri yok; bu koşuyu başlatmak "
        "boşa GPU yakar."
    )

share = supervised_tokens / max(1, total_tokens)
print(f"kabul  : {len(kept)}")
print(
    f"atılan: {dropped_truncated} (bağlamda kesilmiş), "
    f"{dropped_no_supervision} (asistan turn'üne ulaşılamadı), "
    f"{dropped_invalid} (geçersiz)"
)
print(f"token  : {total_tokens} toplam, {supervised_tokens} denetimli ({share:.1%})")
average = total_tokens // len(kept)
print(f"uzunluk: ortalama {average}, en uzun {longest} (sınır {MAX_TOKENS_PER_RECORD})")

In [ ]:
# @title 9 — Token formatını doğrula (eğitimden ÖNCE)
#
# Maske bozuksa model araç çıktısı uydurmayı öğrenir ve kayıp normal görünür. Bu
# kontrol dakikalar sürer, bir 4B epoch saatler sürer. Sıra burada.

from gotooltrain import assert_mask_sane
from gotooltrain.evalstore import sha256_text
from gotooltrain.template import load_template_source

# 8. hücrede ölçülen kayıt; eğitimin de göreceği kayıt. Render burada tekrar
# yapılmıyor: 8. hücre zaten `first_rendered`'ı bıraktı.
example = first_rendered
assert_mask_sane(example)

print("maske tutarlı: labels ve assistant_mask aynı yeri işaretliyor")
print("denetimli token:", example.supervised_tokens, "/", len(example.input_ids))
print("template sha256:", sha256_text(load_template_source())[:16])
print("ilk 300 karakter:")
print(tokenizer.decode(example.input_ids[:120], skip_special_tokens=False)[:300])

In [ ]:
# @title 10 — Eğitimi başlat (periyodik Hub yüklemesiyle)
#
# Adafactor: ilk momenti tutmaz, ikinci momenti çarpanlaştırır. Optimizer durumu
# birkaç GB yerine birkaç MB — tek kartta 4B'yi sığdıran şey bu.
#
# **--gradient-checkpointing burada belirleyici.** Bu modelde 32 katmanın 24'ü
# Gated DeltaNet; duram [32 v_heads, 128, 128] yani token başına 1 MB. Aktivasyon
# belleği bağlamla doğrusal büyür. Checkpointing olmadan 4096 token ~98 GB, 8192
# ~196 GB. Etkinleşmesi için modelin *train* modunda olması gerekir
# (`from_pretrained` eval modunda döndürür) — bu, sürümde artık düzeltildi.
# Çalışma çıktısında "gradient checkpointing on: N module(s)" satırını **gör**;
# yoksa bayrak hiç çalışmamış demektir.
#
# --hub-push-every: optimizer ADIMI sayar, batch değil. 8 kademe biriktirmeli
# olduğu için "her batch'te gönder" demek accumulation ayarına bağımlı olurdu.
#
# İlk koşuda DRY_RUN=True ile başla: pahalı hücrelerden geçer, takvimi ve
# dosyer yazımını prova eder, tek bir bayt göndermez. Yeşil görünce DRY_RUN=False
# yapıp yeniden çalıştır.

import subprocess

OUTPUT = "runs/colab-sft"

with open("data/sft.jsonl", "w", encoding="utf-8", newline="\n") as handle:
    for row in kept:
        handle.write(json.dumps(row) + "\n")
print(f"data/sft.jsonl yazıldı: {len(kept)} kayıt (mesaj biçimi)")

command = [
    sys.executable, "-m", "gotooltrain.traincli", "sft",
    "--model", MODEL_ID,
    "--output", OUTPUT,
    "--tokenizer", MODEL_ID,
    "--dataset", "data/sft.jsonl",
    "--dtype", "bfloat16",
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--grad-accum", str(GRAD_ACCUM),
    "--lr", str(LEARNING_RATE),
    "--max-length", str(CONTEXT_LENGTH),
    "--optimizer", "adafactor",
    "--gradient-checkpointing",
    "--hub-repo-id", HF_REPO_ID,
    "--hub-push-every", str(PUSH_EVERY),
]
if DRY_RUN:
    command.append("--hub-dry-run")

print(" ".join(command))
print()
print("Bu hücre saatlerce sürebilir. Yayınlar PUSH_EVERY adımda birer olur,")
print("böylece sekme kapanırsa kaybedilen şey en fazla PUSH_EVERY adım olur.")
print()
result = None
log_path = pathlib.Path("train.log")
with log_path.open("w", encoding="utf-8", buffering=1) as log:
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
        log.write(line)
    result = process.wait()

print("exit:", result)

# Hata, hücre çıktısında **kaybolmasın diye** ayrıca dosyadan basılıyor. Önceki
# koşuda alt sürecin "error: ..." satırı hiç görünmedi; iki hücre sonra "hub_push.json
# yok" diye yanlış yere baktık. Uzun bir koşunun çıktısı kaydırılınca kaybolabilir,
# yani teşhisin tesadüfe bağlı olmaması gerekiyor.
if result != 0:
    out = pathlib.Path(OUTPUT)
    if not out.is_dir():
        cause = "Çıktı dizini hiç oluşmadı: koşu modeli yüklenmeden ya da veriyi okuyamadan durdu."
    else:
        listing = sorted(p.name for p in out.iterdir())
        print(f"\n{OUTPUT} içinde: {listing}")
        if "training_plan.json" not in listing:
            cause = "Plan yazılmadı: koşu başlamadan durdu."
        elif "hub_push.json" not in listing:
            cause = "Plan yazıldı ama bitmedi; yayınlama adımına hiç gelinmedi."
        else:
            cause = "Yayınladı ama hata ile bitti."
    tail = log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-40:]
    print("\n--- train.log (son 40 satır) ---")
    for line in tail:
        print(" ", line)
    raise SystemExit(
        f"Eğitim {result} ile bitti. Yukarıdaki çıktıya bakın. {cause} "
        "Bu hücre hatayı yutmaz: 11. hücreye geçmeden durur."
    )

In [ ]:
# @title 11 — Ne olduğunu doğrula
#
# "Push ettim" demek yetmez; push'un *ne* olduğu okunmalı. Yayınlanan klasör
# kendi kökenini taşır: hangi adım, hangi ayarlar.
#
# Ama önce ayrım: çıktı dizini hiç oluşmadı mı, yoksa koştu ve yayınlama adımına
# mı gelmedi? Bu ikisi farklı hatalar ve aynı hata mesajıyla gelmez. "hub_push.json
# yok" demek, asıl hatayı değil *sonucunu* söyler; 10. hücre artık yutmuyor ama
# yine de teşhis burada kesinleşmeli.

import json
import pathlib

from gotooltrain import read_push_state

output_dir = pathlib.Path(OUTPUT)
if not output_dir.is_dir():
    raise SystemExit(
        f"{output_dir} yok — eğitim hiç çıktı üretmedi, yani model yüklenmeden ya da "
        "planı yazmadan durdu. 10. hücrenin çıktısına bakın."
    )

listing = sorted(p.name for p in output_dir.iterdir())
print(f"{output_dir} içinde: {listing}")
print()

state = read_push_state(output_dir)
print(f"step {state['step']}/{state['total_steps']}")
plan_record = state["plan"]
for key in (
    "model_id",
    "learning_rate",
    "epochs",
    "context_length",
    "optimizer",
    "token_format",
    "gradient_checkpointing",
):
    print(f"  {key:22} {plan_record[key]}")
print(f"  {'hub':22} {plan_record['hub']}")

if not DRY_RUN:
    from huggingface_hub import HfApi

    api = HfApi()
    files = api.list_repo_files(HF_REPO_ID)
    print(f"\n{HF_REPO_ID} içinde {len(files)} dosya:")
    for name in sorted(files)[:15]:
        print("  ", name)
    print("\ngit log gibi: her push bir commit. Adım commit mesajında.")
else:
    print("\nDRY_RUN=True idi: hiçbir şey gönderilmedi. Yukarıdaki 'pushed step'")
    print("satırlarını 10. hücrede görmüş olmalısınız. Şimdi DRY_RUN=False yapıp")
    print("yeniden çalıştırın.")

## Bu koşu ne ölçer, ne ölçmez

**Ölçer:** token formatı doğrudan gerçek tokenizer ile üretildi; maske `{% generation %}`
bloğundan geldi; loss yalnızca asistan token'larına uygulandı; 4B tam FT gerçekten
bir optimizer adımı attı ve checkpoint yazdı; periyodik yayın takvimi çalıştı.

**Ölçmez:** Go yeteneği. Korpus Go-UT-Bench'in "bu dosya için birim testi yaz"
görevlerinden gelir — tek turlu, araçsız, sekiz depodan biri ağırlıklı. Bu, korpusun
katalog öğretmediğini gösteren bir gerçektir, hata değil. Yetenek ölçümü için
`docs/EVAL.md`'deki üç aşamalı eval gerekir; o konteyner havuzu + `rtk` + gerçek Go
 depoları ister, Colab'da bu yoktur.

## Yayınlama hakkında dürüstlük

- `--hub-dry-run` **hiçbir şey yüklemez**; defter bunu log'da `DRY RUN` diye yazar.
- Gerçek koşuda `--hub-repo-id` verdiyseniz token'ın **yazma yetkisi** olmalı.
  Yazma yetkisi yoksa ilk push'ta hata verir — 700 adım sonra değil, `HF_TOKEN`
  çözümlendiği anda değil (pusher, model yüklenmeden önce kurulur).
- Yayınlanan klasör `hub_push.json` taşır: adım, toplam adım ve tüm hiperparametreler.
  Token **oraya yazılmaz**; sadece değişkenin adı yazılır.
- Bu bir **checkpoint**, tam bir **resume** değildir: optimizer durumu saklanmaz.
  Kaldığınız yerden devam etmek `resume_from` + yeniden eğitim demektir.